# 02 Data quality and cleaning

**Purpose:** check every table against the rules it should follow, explain anything unusual, and save clean, typed tables for the rest of the project.

**Inputs:** the raw data files in `data/raw/`.

**Outputs** (in `data/processed/`):
- `orders.parquet`: train and test orders in one table, with a `split` column
- `legs.parquet`: train and test route legs in one table
- `outlets.parquet`, `vehicles.parquet`, `calendar.parquet`, `road_conditions.parquet`, `traffic_speed.parquet`, `district_travel.parquet`, `service_allowance.parquet`

**Approach:** every rule is written as a check that reports how many rows break it. One summary table shows everything at once, instead of the notebook stopping at the first problem. Nothing is dropped silently: each fix or exclusion is stated with its reason.

In [1]:
import numpy as np
import pandas as pd

from xora.io import load_raw, save_processed
from xora.timeutils import hhmm_to_minutes
from xora import checks as ck

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

In [2]:
orders = load_raw("deliveries_train.csv")
legs = load_raw("route_legs_train.csv")
orders_test = load_raw("task1_test_inputs.csv")
legs_test = load_raw("route_legs_test.csv")
outlets = load_raw("outlets.csv")
vehicles = load_raw("vehicles.csv")
calendar = load_raw("calendar.csv")
road = load_raw("road_conditions.csv")
traffic = load_raw("traffic_speed.csv")
district_travel = load_raw("district_travel.csv")
allowance = load_raw("service_allowance.csv")

## 1. Structural checks

Keys must be unique, categories must take only their documented values, and quantities must be positive.

In [3]:
BRANDS = {"Fresh", "Style", "Tech"}
DEPOTS = {"Peliyagoda", "Kandy"}

rows = []
for name, df in [("orders train", orders), ("orders test", orders_test)]:
    rows += [
        ck.unique_key(df, ["delivery_id"], name),
        ck.allowed_values(df, "brand", BRANDS, name),
        ck.allowed_values(df, "depot", DEPOTS, name),
        ck.allowed_values(df, "temp_requirement", {"chilled", "ambient"}, name),
        ck.allowed_values(df, "dispatch_status", {"attempted", "deferred", "not_run"}, name),
        ck.check(f"{name}: units, weight and volume are positive",
                 (df[["order_units", "order_weight_kg", "order_volume_m3"]] <= 0).any(axis=1).sum(), len(df)),
    ]
for name, df in [("legs train", legs), ("legs test", legs_test)]:
    rows += [
        ck.unique_key(df, ["leg_id"], name),
        ck.unique_key(df, ["route_id", "seq"], name),
        ck.in_range(df, "dow", 0, 5, name),
        ck.in_range(df, "monsoon", 0, 1, name),
        ck.check(f"{name}: distance is positive", (df.distance_km <= 0).sum(), len(df)),
    ]
rows += [
    ck.unique_key(outlets, ["outlet_id"], "outlets"),
    ck.unique_key(vehicles, ["vehicle_id"], "vehicles"),
    ck.unique_key(calendar, ["date"], "calendar"),
    ck.unique_key(road, ["district", "date"], "road conditions"),
    ck.check("train and test share no delivery_id", len(set(orders.delivery_id) & set(orders_test.delivery_id)), len(orders_test)),
    ck.check("test and train share no route_id", legs_test.route_id.isin(legs.route_id).sum(), len(legs_test)),
]
ck.report(rows)

,check,passed,failing_rows,total_rows,note
0,orders train: delivery_id is unique,True,0,92307,
1,"orders train: brand in ['Fresh', 'Style', 'Tech']",True,0,92307,
2,"orders train: depot in ['Kandy', 'Peliyagoda']",True,0,92307,
3,"orders train: temp_requirement in ['ambient', 'chilled']",True,0,92307,
4,"orders train: dispatch_status in ['attempted', 'deferred', 'not_run']",True,0,92307,
5,"orders train: units, weight and volume are positive",True,0,92307,
6,orders test: delivery_id is unique,True,0,5014,
7,"orders test: brand in ['Fresh', 'Style', 'Tech']",True,0,5014,
8,"orders test: depot in ['Kandy', 'Peliyagoda']",True,0,5014,
9,"orders test: temp_requirement in ['ambient', 'chilled']",True,0,5014,


Every structural check passes. Day of week never reaches 6 because Waypoint does not deliver on Sundays.